In [1]:
import pandas as pd
import urllib.request
import zipfile
import os

print("1. Baixando o arquivo bruto oficial do IBGE (isso leva só uns segundinhos)...")
# Link direto do FTP oficial do IBGE com os dados do universo do Censo (SP Interior)
url_ibge = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2010/Resultados_do_Universo/Agregados_por_Setores_Censitarios/SP_Exceto_a_Capital_20171016.zip"
zip_path = "SP_Agregados.zip"
extract_folder = "IBGE_SP"

urllib.request.urlretrieve(url_ibge, zip_path)

print("2. Extraindo os dados...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

# Encontrando o arquivo 'Básico' que contém a variável de renda V005
arquivo_basico = [f for f in os.listdir(extract_folder) if f.startswith('Basico') and f.endswith('.csv')][0]
caminho_csv = os.path.join(extract_folder, arquivo_basico)

print(f"3. Lendo os dados e filtrando Campinas...")
# Lendo o CSV (o IBGE usa separador ';' e vírgula para decimais)
df_sp = pd.read_csv(caminho_csv, sep=';', encoding='latin1', decimal=',')

# O código de Campinas no IBGE é 3509502
df_campinas = df_sp[df_sp['Cod_municipio'] == 3509502].copy()

# A coluna V005 é o "Valor do rendimento nominal médio mensal das pessoas responsáveis"
df_campinas['V005'] = pd.to_numeric(df_campinas['V005'], errors='coerce')

print("4. Agrupando a renda por região...")
# Criando a região a partir dos 10 primeiros dígitos do código do setor
df_campinas['Cod_setor_str'] = df_campinas['Cod_setor'].astype(str)
df_campinas['regiao_ibge'] = df_campinas['Cod_setor_str'].str[:10]

# Agrupando para ter a média de renda por região
df_renda_regiao = df_campinas.groupby('regiao_ibge').agg(
    total_setores=('Cod_setor', 'count'),
    renda_media_bairro=('V005', 'mean')
).reset_index()

print("\n✅ SUCESSO! Renda Média por Região em Campinas:")
display(df_renda_regiao.sort_values(by='renda_media_bairro', ascending=False))

1. Baixando o arquivo bruto oficial do IBGE (isso leva só uns segundinhos)...


HTTPError: HTTP Error 404: Not Found